In [5]:
# Day 2 - Sales Analytics Without Pandas
# Squad Python Specialization - Data Science, AI & Analytics

# Tasks 1-7 use only the standard library. pandas is allowed in task 8, to check your work.

import csv
from pathlib import Path

DATA_PATH = Path("data") / "electronics_sales.csv"


def read_sales(path: Path = DATA_PATH) -> list[dict[str, str]]:
    # """Read the CSV with the csv module and return the raw rows."""
     with open(path, mode="r", newline="", encoding="utf-8") as file:
        reader = csv.DictReader(file)
        rows = list(reader)

        print(f"Number of rows found: {len(rows)}")
        print(f"Column names found: {reader.fieldnames}")

        return rows
read_sales(DATA_PATH)
    # raise NotImplementedError

Number of rows found: 50
Column names found: ['sale_id', 'sale_date', 'store', 'product', 'quantity', 'unit_price_kes', 'payment_method']


[{'sale_id': 'S-3001',
  'sale_date': '2026-09-01',
  'store': 'Westgate',
  'product': 'Safaricom Neon Ray',
  'quantity': '1',
  'unit_price_kes': '5999',
  'payment_method': 'Cash'},
 {'sale_id': 'S-3002',
  'sale_date': '2026-09-02',
  'store': 'Westgate',
  'product': 'USB-C Charger 25W',
  'quantity': '2',
  'unit_price_kes': '1499',
  'payment_method': 'M-Pesa'},
 {'sale_id': 'S-3003',
  'sale_date': '2026-09-02',
  'store': 'Kisumu Mega Plaza',
  'product': 'Lenovo Tab M10',
  'quantity': '2',
  'unit_price_kes': '24999',
  'payment_method': 'Cash'},
 {'sale_id': 'S-3004',
  'sale_date': '2026-09-02',
  'store': 'Westgate',
  'product': 'Xiaomi Redmi Buds',
  'quantity': '2',
  'unit_price_kes': '3199',
  'payment_method': 'M-Pesa'},
 {'sale_id': 'S-3005',
  'sale_date': '2026-09-03',
  'store': 'Nakuru Westside',
  'product': 'Tecno Spark 20',
  'quantity': '2',
  'unit_price_kes': '13499',
  'payment_method': 'M-Pesa'},
 {'sale_id': 'S-3006',
  'sale_date': '2026-09-03',
  's

In [10]:
def parse_number(value: str | None) -> float | None:
    """Convert a messy numeric string to a float, or return None if invalid."""
    if value is None:
        return None

    cleaned = value.strip().upper().replace("KES", "").strip()
    cleaned = cleaned.replace(",", "")

    word_to_num = {
        "ZERO": 0,
        "ONE": 1,
        "TWO": 2,
        "THREE": 3,
        "FOUR": 4,
        "FIVE": 5,
        "SIX": 6,
        "SEVEN": 7,
        "EIGHT": 8,
        "NINE": 9,
        "TEN": 10
    }

    if cleaned in ("", "N/A", "NONE", "NULL", "-"):
        return None

    if cleaned in word_to_num:
        return word_to_num[cleaned]


    try:
        return float(cleaned)
    except ValueError:
        return None


print(parse_number("two"))  # Returns: 2.0
print(parse_number("  KES 1,200.50  "))  # Returns: 1200.50
print(parse_number("    N/A"))  # Returns: None

    
    # """Convert a messy numeric string to a float, or return None if it cannot be trusted."""
    # raise NotImplementedError

2
1200.5
None


In [14]:
def convert_rows(rows: list[dict[str, str]]) -> tuple[list[dict], list[dict]]:
    """Return clean rows and skipped rows with reasons."""
    clean_rows = []
    skipped_rows = []

    def find_value(row: dict[str, str], *names: str) -> str | None:
        normalized = {
            key.lower().replace("_", "").replace(" ", ""): value
            for key, value in row.items()
        }

        for name in names:
            target = name.lower().replace("_", "").replace(" ", "")
            # FIX: Match exact string OR fallback to a substring lookup (e.g., 'unitprice' matches 'unitpricekes')
            if target in normalized:
                return normalized[target]
            for key, value in normalized.items():
                if target in key:
                    return value
        return None

    for row_number, row in enumerate(rows, start=1):
        quantity_raw = find_value(row, "quantity", "qty")
        unit_price_raw = find_value(row, "unit_price", "unit price", "price")
        revenue_raw = find_value(row, "revenue", "total sales", "sales")

        quantity = parse_number(quantity_raw)
        unit_price = parse_number(unit_price_raw)
        revenue = parse_number(revenue_raw)

        if quantity is None:
            skipped_rows.append(
                {"row": row_number, "reason": "Invalid quantity", "data": row}
            )
            continue

        if unit_price is None:
            skipped_rows.append(
                {"row": row_number, "reason": "Invalid unit price", "data": row}
            )
            continue

        if revenue is None and (
            revenue_raw is None or not str(revenue_raw).strip()
        ):
            revenue = quantity * unit_price
        elif revenue is None:
            skipped_rows.append(
                {"row": row_number, "reason": "Invalid revenue", "data": row}
            )
            continue

        clean_row = dict(row)
        clean_row["quantity"] = quantity
        clean_row["unit_price"] = unit_price
        clean_row["revenue"] = revenue
        clean_rows.append(clean_row)

    return clean_rows, skipped_rows

In [15]:
def total_revenue(rows: list[dict]) -> float:
    """Calculate total revenue as quantity × unit price."""
    total = 0.0

    for row in rows:
        quantity = row.get("quantity")
        unit_price = row.get("unit_price", row.get("unit_price_ke"))

        if not isinstance(quantity, (int, float)):
            quantity = parse_number(str(quantity))
        if not isinstance(unit_price, (int, float)):
            unit_price = parse_number(str(unit_price))

        if quantity is not None and unit_price is not None:
            total += quantity * unit_price

    return total


print(f"Total Revenue: KES {total_revenue(convert_rows(read_sales())[0]):,.2f}")

Number of rows found: 50
Column names found: ['sale_id', 'sale_date', 'store', 'product', 'quantity', 'unit_price_kes', 'payment_method']
Total Revenue: KES 1,317,210.00
